In [ ]:
import duckdb
import pandas as pd

pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', 20)
pd.set_option('display.width', 120)

con = duckdb.connect(r'data\gold.duckdb', read_only=True)
print('Connected to gold.duckdb')

def q(sql):
    return con.sql(sql).df()

In [ ]:
# ── Change these to analyse any player ──────────────────────────────────────
PLAYER = "V Kohli"
FORMAT = "odi"   # for format-specific queries: t20, odi, test, it20, odm, mdm
# ────────────────────────────────────────────────────────────────────────────

## Data Coverage

In [ ]:
# Matches by type
q("""
SELECT match_type, count(*) AS matches
FROM main_marts.fct_match_results
GROUP BY match_type
ORDER BY matches DESC
""")

In [ ]:
# Matches per year (last 10 years)
q("""
SELECT year, match_type, count(*) AS matches
FROM main_marts.fct_match_results
WHERE year >= year(today()) - 10
GROUP BY year, match_type
ORDER BY year DESC, matches DESC
""")

In [ ]:
# Most recent matches loaded
q("""
SELECT match_id, match_type, match_date, team_1, team_2, outcome_winner
FROM main_marts.fct_match_results
ORDER BY match_date DESC
LIMIT 20
""")

## Batting

In [ ]:
# Top 20 ODI run scorers (min 50 innings)
q("""
SELECT player_name, matches, innings, total_runs, highest_score,
       batting_avg, batting_sr, centuries, fifties
FROM main_marts.mart_batting_career
WHERE match_type = 'odi'
  AND innings >= 50
ORDER BY total_runs DESC
LIMIT 20
""")

In [ ]:
# Career stats across all formats for selected player
q(f"""
SELECT player_name, match_type, matches, innings, total_runs,
       batting_avg, batting_sr, centuries, fifties
FROM main_marts.mart_batting_career
WHERE player_name = '{PLAYER}'
ORDER BY match_type
""")

In [ ]:
# Batting average trend by year for selected player and format
q(f"""
SELECT year,
       count(distinct match_id) AS matches,
       sum(runs)                AS runs,
       round(sum(runs)::double / nullif(sum(dismissals), 0), 2) AS avg
FROM main_intermediate.int_batting_by_innings
WHERE player_name = '{PLAYER}'
  AND match_type  = '{FORMAT}'
  AND NOT super_over
GROUP BY year
ORDER BY year
""")

In [ ]:
# Batting average trend by year for a player (ODIs)
q("""
SELECT year,
       count(distinct match_id) AS matches,
       sum(runs) AS runs,
       round(sum(runs)::double / nullif(sum(dismissals), 0), 2) AS avg
FROM main_intermediate.int_batting_by_innings
WHERE player_name = 'V Kohli'
  AND match_type = 'odi'
  AND NOT super_over
GROUP BY year
ORDER BY year
""")

## Bowling

In [ ]:
# Top 20 ODI wicket takers (min 50 innings bowled)
q("""
SELECT player_name, matches, innings_bowled, total_wickets,
       bowling_avg, economy, bowling_sr, five_wicket_hauls
FROM main_marts.mart_bowling_career
WHERE match_type = 'odi'
  AND innings_bowled >= 50
ORDER BY total_wickets DESC
LIMIT 20
""")

In [ ]:
# Best T20 economy rates (min 50 innings)
q("""
SELECT player_name, innings_bowled, total_wickets, economy, bowling_avg, bowling_sr
FROM main_marts.mart_bowling_career
WHERE match_type = 't20'
  AND innings_bowled >= 50
ORDER BY economy
LIMIT 20
""")

In [ ]:
# Economy by over number in T20s (powerplay vs middle vs death)
q("""
SELECT over_number,
       count(*) AS balls,
       sum(runs_total) AS runs,
       round(sum(runs_total) * 6.0 / nullif(
           count(*) - sum(extras_wides::integer), 0), 2) AS economy
FROM main_marts.fct_deliveries
WHERE match_type = 't20'
  AND innings_number IN (1, 2)
GROUP BY over_number
ORDER BY over_number
""")

## Match Results

In [ ]:
# India ODI win/loss record
q("""
SELECT team_1 AS team, count(*) AS played,
       sum((outcome_winner = team_1)::integer) AS won,
       sum((outcome_winner = team_2)::integer) AS lost,
       sum(is_tie::integer) AS tied,
       sum((NOT has_result)::integer) AS no_result,
       round(sum((outcome_winner = team_1)::integer) * 100.0
           / nullif(sum(has_result::integer), 0), 1) AS win_pct
FROM main_marts.fct_match_results
WHERE match_type = 'odi'
  AND team_1 = 'India'
UNION ALL
SELECT team_2, count(*),
       sum((outcome_winner = team_2)::integer),
       sum((outcome_winner = team_1)::integer),
       sum(is_tie::integer),
       sum((NOT has_result)::integer),
       round(sum((outcome_winner = team_2)::integer) * 100.0
           / nullif(sum(has_result::integer), 0), 1)
FROM main_marts.fct_match_results
WHERE match_type = 'odi'
  AND team_2 = 'India'
""")

In [ ]:
# India vs Australia head-to-head (all formats)
q("""
SELECT match_type, count(*) AS played,
       sum((outcome_winner = 'India')::integer) AS india_wins,
       sum((outcome_winner = 'Australia')::integer) AS aus_wins,
       sum(is_tie::integer) AS ties
FROM main_marts.fct_match_results
WHERE has_result
  AND (
    (team_1 = 'India' AND team_2 = 'Australia') OR
    (team_1 = 'Australia' AND team_2 = 'India')
  )
GROUP BY match_type
ORDER BY match_type
""")

In [ ]:
# Toss advantage: does winning the toss help?
q("""
SELECT match_type, toss_decision, count(*) AS matches,
       sum((toss_winner = outcome_winner)::integer) AS toss_winner_won,
       round(sum((toss_winner = outcome_winner)::integer) * 100.0
           / nullif(count(*), 0), 1) AS toss_win_pct
FROM main_marts.fct_match_results
WHERE has_result
  AND match_type IN ('t20', 'odi', 'test')
GROUP BY match_type, toss_decision
ORDER BY match_type, toss_decision
""")

In [ ]:
# Most player of the match awards
q("""
SELECT player_of_match AS player, match_type, count(*) AS awards
FROM main_marts.fct_match_results
WHERE player_of_match IS NOT NULL
GROUP BY player_of_match, match_type
ORDER BY awards DESC
LIMIT 20
""")

In [ ]:
# IPL career batting stats for selected player
q(f"""
WITH bat AS (
    SELECT d.match_id, d.innings_number,
        sum(d.runs_batter)                                                              AS runs,
        sum(CASE WHEN NOT d.extras_wides THEN 1 ELSE 0 END)                            AS balls_faced,
        sum(CASE WHEN d.runs_batter = 4 AND NOT d.runs_non_boundary THEN 1 ELSE 0 END) AS fours,
        sum(CASE WHEN d.runs_batter = 6 AND NOT d.runs_non_boundary THEN 1 ELSE 0 END) AS sixes
    FROM main_marts.fct_deliveries d
    JOIN main_marts.fct_match_results m USING (match_id)
    WHERE d.batter = '{PLAYER}'
      AND m.event_name ILIKE '%indian premier league%'
      AND d.innings_number IN (1, 2)
    GROUP BY d.match_id, d.innings_number
),
dis AS (
    SELECT DISTINCT d.match_id, d.innings_number
    FROM main_marts.fct_deliveries d
    JOIN main_marts.fct_match_results m USING (match_id)
    WHERE d.is_wicket AND d.wicket_player_out = '{PLAYER}'
      AND m.event_name ILIKE '%indian premier league%'
      AND d.innings_number IN (1, 2)
),
all_inn AS (
    SELECT match_id, innings_number FROM bat
    UNION
    SELECT match_id, innings_number FROM dis
    UNION
    SELECT DISTINCT d.match_id, d.innings_number
    FROM main_marts.fct_deliveries d
    JOIN main_marts.fct_match_results m USING (match_id)
    WHERE d.non_striker = '{PLAYER}'
      AND m.event_name ILIKE '%indian premier league%'
      AND d.innings_number IN (1, 2)
      AND NOT EXISTS (
          SELECT 1 FROM main_marts.fct_deliveries d2
          WHERE d2.match_id = d.match_id AND d2.batter = '{PLAYER}'
      )
),
inn AS (
    SELECT
        a.match_id, a.innings_number,
        coalesce(b.runs,        0) AS runs,
        coalesce(b.balls_faced, 0) AS balls_faced,
        coalesce(b.fours,       0) AS fours,
        coalesce(b.sixes,       0) AS sixes,
        (d.match_id IS NOT NULL)::integer AS dismissed
    FROM all_inn a
    LEFT JOIN bat b USING (match_id, innings_number)
    LEFT JOIN dis d USING (match_id, innings_number)
)
SELECT
    '{PLAYER}'                                                  AS player_name,
    count(DISTINCT match_id)                                    AS matches,
    count(*)                                                    AS innings,
    sum(runs)                                                   AS total_runs,
    max(runs)                                                   AS highest_score,
    round(sum(runs)::double / nullif(sum(dismissed), 0), 2)    AS batting_avg,
    round(sum(runs) * 100.0 / nullif(sum(balls_faced), 0), 2)  AS strike_rate,
    sum(CASE WHEN runs >= 100 THEN 1 ELSE 0 END)               AS centuries,
    sum(CASE WHEN runs >= 50 AND runs < 100 THEN 1 ELSE 0 END) AS fifties,
    sum(fours)                                                  AS total_fours,
    sum(sixes)                                                  AS total_sixes
FROM inn
""")

In [ ]:
# IPL stats by season for selected player
q(f"""
WITH bat AS (
    SELECT d.match_id, d.innings_number, m.season,
        sum(d.runs_batter)                                   AS runs,
        sum(CASE WHEN NOT d.extras_wides THEN 1 ELSE 0 END) AS balls_faced
    FROM main_marts.fct_deliveries d
    JOIN main_marts.fct_match_results m USING (match_id)
    WHERE d.batter = '{PLAYER}'
      AND m.event_name ILIKE '%indian premier league%'
      AND d.innings_number IN (1, 2)
    GROUP BY d.match_id, d.innings_number, m.season
),
dis AS (
    SELECT DISTINCT d.match_id, d.innings_number, m.season
    FROM main_marts.fct_deliveries d
    JOIN main_marts.fct_match_results m USING (match_id)
    WHERE d.is_wicket AND d.wicket_player_out = '{PLAYER}'
      AND m.event_name ILIKE '%indian premier league%'
      AND d.innings_number IN (1, 2)
),
all_inn AS (
    SELECT match_id, innings_number, season FROM bat
    UNION
    SELECT match_id, innings_number, season FROM dis
    UNION
    SELECT DISTINCT d.match_id, d.innings_number, m.season
    FROM main_marts.fct_deliveries d
    JOIN main_marts.fct_match_results m USING (match_id)
    WHERE d.non_striker = '{PLAYER}'
      AND m.event_name ILIKE '%indian premier league%'
      AND d.innings_number IN (1, 2)
      AND NOT EXISTS (
          SELECT 1 FROM main_marts.fct_deliveries d2
          WHERE d2.match_id = d.match_id AND d2.batter = '{PLAYER}'
      )
),
inn AS (
    SELECT
        a.match_id, a.innings_number, a.season,
        coalesce(b.runs,        0) AS runs,
        coalesce(b.balls_faced, 0) AS balls_faced,
        (d.match_id IS NOT NULL)::integer AS dismissed
    FROM all_inn a
    LEFT JOIN bat b USING (match_id, innings_number)
    LEFT JOIN dis d USING (match_id, innings_number)
)
SELECT
    season,
    count(DISTINCT match_id)                                   AS matches,
    count(*)                                                   AS innings,
    sum(runs)                                                  AS runs,
    max(runs)                                                  AS highest,
    round(sum(runs)::double / nullif(sum(dismissed), 0), 2)   AS avg,
    round(sum(runs) * 100.0 / nullif(sum(balls_faced), 0), 2) AS sr,
    sum(CASE WHEN runs >= 50 THEN 1 ELSE 0 END)               AS fifties_plus
FROM inn
GROUP BY season
ORDER BY season
""")

In [ ]:
# IPL innings log for selected player (most recent first)
q(f"""
WITH bat AS (
    SELECT d.match_id, d.innings_number, d.batting_team, d.bowling_team, d.match_date,
        m.season,
        sum(d.runs_batter)                                                              AS runs,
        sum(CASE WHEN NOT d.extras_wides THEN 1 ELSE 0 END)                            AS balls_faced,
        sum(CASE WHEN d.runs_batter = 4 AND NOT d.runs_non_boundary THEN 1 ELSE 0 END) AS fours,
        sum(CASE WHEN d.runs_batter = 6 AND NOT d.runs_non_boundary THEN 1 ELSE 0 END) AS sixes
    FROM main_marts.fct_deliveries d
    JOIN main_marts.fct_match_results m USING (match_id)
    WHERE d.batter = '{PLAYER}'
      AND m.event_name ILIKE '%indian premier league%'
      AND d.innings_number IN (1, 2)
    GROUP BY d.match_id, d.innings_number, d.batting_team, d.bowling_team, d.match_date, m.season
),
dis AS (
    SELECT DISTINCT d.match_id, d.innings_number, d.batting_team, d.bowling_team, d.match_date,
        m.season
    FROM main_marts.fct_deliveries d
    JOIN main_marts.fct_match_results m USING (match_id)
    WHERE d.is_wicket AND d.wicket_player_out = '{PLAYER}'
      AND m.event_name ILIKE '%indian premier league%'
      AND d.innings_number IN (1, 2)
),
all_inn AS (
    SELECT match_id, innings_number, batting_team, bowling_team, match_date, season FROM bat
    UNION
    SELECT match_id, innings_number, batting_team, bowling_team, match_date, season FROM dis
    UNION
    SELECT DISTINCT d.match_id, d.innings_number, d.batting_team, d.bowling_team, d.match_date,
        m.season
    FROM main_marts.fct_deliveries d
    JOIN main_marts.fct_match_results m USING (match_id)
    WHERE d.non_striker = '{PLAYER}'
      AND m.event_name ILIKE '%indian premier league%'
      AND d.innings_number IN (1, 2)
      AND NOT EXISTS (
          SELECT 1 FROM main_marts.fct_deliveries d2
          WHERE d2.match_id = d.match_id AND d2.batter = '{PLAYER}'
      )
)
SELECT
    a.season, a.match_date, a.batting_team,
    a.bowling_team                                                               AS vs,
    coalesce(b.runs,        0)                                                   AS runs,
    coalesce(b.balls_faced, 0)                                                   AS balls_faced,
    round(coalesce(b.runs, 0) * 100.0 / nullif(coalesce(b.balls_faced, 0), 0), 1) AS strike_rate,
    coalesce(b.fours, 0)                                                         AS fours,
    coalesce(b.sixes, 0)                                                         AS sixes,
    (d.match_id IS NULL)                                                         AS not_out
FROM all_inn a
LEFT JOIN bat b USING (match_id, innings_number)
LEFT JOIN dis d USING (match_id, innings_number)
ORDER BY a.match_date DESC
""")

In [ ]:
# V Kohli IPL innings log (most recent first)
q("""
WITH bat AS (
    SELECT d.match_id, d.innings_number, d.batting_team, d.bowling_team, d.match_date,
        m.season,
        sum(d.runs_batter)                                                              AS runs,
        sum(CASE WHEN NOT d.extras_wides THEN 1 ELSE 0 END)                            AS balls_faced,
        sum(CASE WHEN d.runs_batter = 4 AND NOT d.runs_non_boundary THEN 1 ELSE 0 END) AS fours,
        sum(CASE WHEN d.runs_batter = 6 AND NOT d.runs_non_boundary THEN 1 ELSE 0 END) AS sixes
    FROM main_marts.fct_deliveries d
    JOIN main_marts.fct_match_results m USING (match_id)
    WHERE d.batter = 'V Kohli'
      AND m.event_name ILIKE '%indian premier league%'
      AND d.innings_number IN (1, 2)
    GROUP BY d.match_id, d.innings_number, d.batting_team, d.bowling_team, d.match_date, m.season
),
dis AS (
    SELECT DISTINCT d.match_id, d.innings_number, d.batting_team, d.bowling_team, d.match_date,
        m.season
    FROM main_marts.fct_deliveries d
    JOIN main_marts.fct_match_results m USING (match_id)
    WHERE d.is_wicket AND d.wicket_player_out = 'V Kohli'
      AND m.event_name ILIKE '%indian premier league%'
      AND d.innings_number IN (1, 2)
),
all_inn AS (
    SELECT match_id, innings_number, batting_team, bowling_team, match_date, season FROM bat
    UNION
    SELECT match_id, innings_number, batting_team, bowling_team, match_date, season FROM dis
    UNION
    SELECT DISTINCT d.match_id, d.innings_number, d.batting_team, d.bowling_team, d.match_date,
        m.season
    FROM main_marts.fct_deliveries d
    JOIN main_marts.fct_match_results m USING (match_id)
    WHERE d.non_striker = 'V Kohli'
      AND m.event_name ILIKE '%indian premier league%'
      AND d.innings_number IN (1, 2)
      AND NOT EXISTS (
          SELECT 1 FROM main_marts.fct_deliveries d2
          WHERE d2.match_id = d.match_id AND d2.batter = 'V Kohli'
      )
)
SELECT
    a.season, a.match_date, a.batting_team,
    a.bowling_team                                                               AS vs,
    coalesce(b.runs,        0)                                                   AS runs,
    coalesce(b.balls_faced, 0)                                                   AS balls_faced,
    round(coalesce(b.runs, 0) * 100.0 / nullif(coalesce(b.balls_faced, 0), 0), 1) AS strike_rate,
    coalesce(b.fours, 0)                                                         AS fours,
    coalesce(b.sixes, 0)                                                         AS sixes,
    (d.match_id IS NULL)                                                         AS not_out
FROM all_inn a
LEFT JOIN bat b USING (match_id, innings_number)
LEFT JOIN dis d USING (match_id, innings_number)
ORDER BY a.match_date DESC
""")

In [ ]:
# IPL dismissal breakdown by season for selected player (diagnostic)
q(f"""
WITH bat AS (
    SELECT d.match_id, d.innings_number, m.season,
        sum(d.runs_batter) AS runs
    FROM main_marts.fct_deliveries d
    JOIN main_marts.fct_match_results m USING (match_id)
    WHERE d.batter = '{PLAYER}'
      AND m.event_name ILIKE '%indian premier league%'
      AND d.innings_number IN (1, 2)
    GROUP BY d.match_id, d.innings_number, m.season
),
dis AS (
    SELECT DISTINCT d.match_id, d.innings_number, m.season
    FROM main_marts.fct_deliveries d
    JOIN main_marts.fct_match_results m USING (match_id)
    WHERE d.is_wicket AND d.wicket_player_out = '{PLAYER}'
      AND m.event_name ILIKE '%indian premier league%'
      AND d.innings_number IN (1, 2)
),
all_inn AS (
    SELECT match_id, innings_number, season FROM bat
    UNION
    SELECT match_id, innings_number, season FROM dis
    UNION
    SELECT DISTINCT d.match_id, d.innings_number, m.season
    FROM main_marts.fct_deliveries d
    JOIN main_marts.fct_match_results m USING (match_id)
    WHERE d.non_striker = '{PLAYER}'
      AND m.event_name ILIKE '%indian premier league%'
      AND d.innings_number IN (1, 2)
      AND NOT EXISTS (
          SELECT 1 FROM main_marts.fct_deliveries d2
          WHERE d2.match_id = d.match_id AND d2.batter = '{PLAYER}'
      )
),
inn AS (
    SELECT
        a.match_id, a.innings_number, a.season,
        coalesce(b.runs, 0)                AS runs,
        (d.match_id IS NOT NULL)::integer  AS dismissed
    FROM all_inn a
    LEFT JOIN bat b USING (match_id, innings_number)
    LEFT JOIN dis d USING (match_id, innings_number)
)
SELECT
    season,
    count(*)                                                        AS innings,
    sum(dismissed)                                                  AS dismissed,
    sum(1 - dismissed)                                              AS not_out,
    sum(runs)                                                       AS runs,
    round(sum(runs)::double / nullif(sum(dismissed), 0), 2)        AS avg
FROM inn
GROUP BY season
ORDER BY season
""")

In [ ]:
# Diagnose avg — show innings, dismissed, not_out per season (V Kohli)
q("""
WITH bat AS (
    SELECT d.match_id, d.innings_number, m.season,
        sum(d.runs_batter) AS runs
    FROM main_marts.fct_deliveries d
    JOIN main_marts.fct_match_results m USING (match_id)
    WHERE d.batter = 'V Kohli'
      AND m.event_name ILIKE '%indian premier league%'
      AND d.innings_number IN (1, 2)
    GROUP BY d.match_id, d.innings_number, m.season
),
dis AS (
    SELECT DISTINCT d.match_id, d.innings_number, m.season
    FROM main_marts.fct_deliveries d
    JOIN main_marts.fct_match_results m USING (match_id)
    WHERE d.is_wicket AND d.wicket_player_out = 'V Kohli'
      AND m.event_name ILIKE '%indian premier league%'
      AND d.innings_number IN (1, 2)
),
all_inn AS (
    SELECT match_id, innings_number, season FROM bat
    UNION
    SELECT match_id, innings_number, season FROM dis
    UNION
    SELECT DISTINCT d.match_id, d.innings_number, m.season
    FROM main_marts.fct_deliveries d
    JOIN main_marts.fct_match_results m USING (match_id)
    WHERE d.non_striker = 'V Kohli'
      AND m.event_name ILIKE '%indian premier league%'
      AND d.innings_number IN (1, 2)
      AND NOT EXISTS (
          SELECT 1 FROM main_marts.fct_deliveries d2
          WHERE d2.match_id = d.match_id AND d2.batter = 'V Kohli'
      )
),
inn AS (
    SELECT
        a.match_id, a.innings_number, a.season,
        coalesce(b.runs, 0)                AS runs,
        (d.match_id IS NOT NULL)::integer  AS dismissed
    FROM all_inn a
    LEFT JOIN bat b USING (match_id, innings_number)
    LEFT JOIN dis d USING (match_id, innings_number)
)
SELECT
    season,
    count(*)                                                        AS innings,
    sum(dismissed)                                                  AS dismissed,
    sum(1 - dismissed)                                              AS not_out,
    sum(runs)                                                       AS runs,
    round(sum(runs)::double / nullif(sum(dismissed), 0), 2)         AS avg
FROM inn
GROUP BY season
ORDER BY season
""")